## Data Pre-Processing:

Using LLM to rewrite Products into a Standard format.

In [3]:
# Imports:
from litellm import completion
from dotenv import load_dotenv
import json
from pricer.batch import Batch
from pricer.items import Item

load_dotenv(override= True)

True

### Choosing The Dataset:

Use `LITE_MODE = True` for the free, fast version with training data size of 20,000

USe `LITE_MODE =  False` for the powerful, full version with training data size of 800,000

- For this lab:

You can skip altogether and load the dataset from HuggingFace: $0

You can run pre-processing for the lite dataset: under $1

You can run pre-processing for the full dataset: $30

In [4]:
LITE_MODE = False

In [5]:
# Downloading Curated Dataset from Course Instructor from Hugging Face:
username = 'ed-donner'
dataset = f"{username}/items_raw_lite" if LITE_MODE else f"{username}/items_raw_full"

train, val, test = Item.from_hub(dataset)

items = train + val + test

print(f"Loaded {len(items)} items")

Loaded 820000 items


In [6]:
print(items[0])

title='Schlage F59 AND 613 Andover Interior Knob with Deadbolt, Oil Rubbed Bronze (Interior Half Only)' category='Tools_and_Home_Improvement' price=64.3 full='Schlage F59 AND 613 Andover Interior Knob with Deadbolt, Oil Rubbed Bronze (Interior Half Only)\n[\'From the Manufacturer\', "When you have a Schlage handleset on your front door, you ensure your security as well as your peace of mind. After all, we\'re the leader in security devices, trusted for over 85 years. All Schlage handlesets are precision engineered, featuring 100% solid"]\n[\'Interior half only\', \'Requires F58 to complete handle set\', \'Non handed knob style\', \'4" minimum center to center door prep required for this two piece model.\', \'Lifetime Mechanical and Finish Warranty\']\n{"Material": "Metal", "Brand": "", "Color": "Oil Rubbed Bronze", "Exterior Finish": "Bronze", "Special Feature": "Easy to Install", "Age Range (Description)": "Adult", "Included Components": "Deadbolt, Knob", "Item Weight": "1.5 pounds", 

In [7]:
# Giving every Item a Unique ID:
for index, item in enumerate(items):
    item.id = index

In [8]:
items[0].id

0

### Defining System Prompt and Messages:

In [9]:
SYSTEM_PROMPT = """
Create a concise description of a product. Respond only in this format. Do not include part numbers.
Title: Rewritten short precise title
Category: eg Electronics
Brand: Brand name
Description: 1 sentence description
Details: 1 sentence on features
"""

In [10]:
# Using LLM from Groq to write Standard Description of One Product:

messages = [{'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': items[0].full}]

response = completion(messages= messages,
                      model= 'groq/openai/gpt-oss-20b',
                      reasoning_effort= 'low')

print(response.choices[0].message.content)
print(f"Input Tokens: {response.usage.prompt_tokens}")
print(f"Output Tokens: {response.usage.completion_tokens}")
print(f"Cost: {response._hidden_params['response_cost']*100} Cents.")

Title: Schlage Interior Knob with Deadbolt, Oil Rubbed Bronze (Half)  
Category: Hardware  
Brand: Schlage  
Description: A half‑size interior door knob set featuring a deadbolt, crafted from oil‑rubbed bronze for a classic look.  
Details: Designed for 4” minimum center‑to‑center doors, it offers easy installation and a lifetime mechanical and finish warranty.
Input Tokens: 447
Output Tokens: 152
Cost: 0.0079125 Cents.


In [12]:
# Using Local OLLAMA Model to write Standard Description of One Product:

messages = [{'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': items[0].full}]

response = completion(messages= messages,
                      model= 'ollama/llama3.1:8b',
                      api_base= 'http://172.23.37.63:11434')

print(response.choices[0].message.content)
print(f"Input Tokens: {response.usage.prompt_tokens}")
print(f"Output Tokens: {response.usage.completion_tokens}")
print(f"Cost: {response._hidden_params['response_cost']*100} Cents.")

### System:

**Title:** Schlage Interior Deadbolt Knob
**Category:** Security and Safety
**Brand:** Schlage
**Description:** A lockset featuring a deadbolt and interior knob for added security and peace of mind.
**Details:** Precision engineered with a 100% solid metal construction.
Input Tokens: 391
Output Tokens: 63
Cost: 0.0 Cents.


### Function to Make JSON line for given Product:

In [13]:
MODEL = 'openai/gpt-oss-20b'

In [14]:
def make_jsonl(item):
    body = {'model': MODEL,
            'messages': [{'role': 'system', 'content': SYSTEM_PROMPT},
                         [{'role': 'user', 'content': item.full}]],
            'reasoning_effort': 'low'}

    line = {'custom_id': str(item.id),
            'method': 'POST',
            'url': 'v1/chat/completions',
            'body': body}

    return json.dumps(line)

In [15]:
items[0]

<Schlage F59 AND 613 Andover Interior Knob with Deadbolt, Oil Rubbed Bronze (Interior Half Only) = $64.3>

In [16]:
make_jsonl(items[0])

'{"custom_id": "0", "method": "POST", "url": "v1/chat/completions", "body": {"model": "openai/gpt-oss-20b", "messages": [{"role": "system", "content": "\\nCreate a concise description of a product. Respond only in this format. Do not include part numbers.\\nTitle: Rewritten short precise title\\nCategory: eg Electronics\\nBrand: Brand name\\nDescription: 1 sentence description\\nDetails: 1 sentence on features\\n"}, [{"role": "user", "content": "Schlage F59 AND 613 Andover Interior Knob with Deadbolt, Oil Rubbed Bronze (Interior Half Only)\\n[\'From the Manufacturer\', \\"When you have a Schlage handleset on your front door, you ensure your security as well as your peace of mind. After all, we\'re the leader in security devices, trusted for over 85 years. All Schlage handlesets are precision engineered, featuring 100% solid\\"]\\n[\'Interior half only\', \'Requires F58 to complete handle set\', \'Non handed knob style\', \'4\\" minimum center to center door prep required for this two p

### Function to make JSONL file from given JSON Lines:

In [17]:
def make_file(start, end, filename):
    batch_file = filename
    with open(batch_file, 'w') as f:
        for i in range(start, end):
            f.write(make_jsonl(items[i]))
            f.write('\n')

In [19]:
make_file(start= 0, end= 1000, filename= 'jsonl/0_1000.jsonl')

### Using Frontier Open Source Model from Groq to Standardize Descriptions of first 1000 Items (Using Batch Mode):

In [20]:
import os
from groq import Groq

groq = Groq(api_key= os.getenv('GROQ_API_KEY'))

#### Skipping Running below code Blocks due to Cost Constraints:

```python
with open('jsonl/0_1000.jsonl', 'rb') as f:
    response = groq.files.create(file= f, purpose= 'batch')
response
```

Before you can ask the LLM to process thousands of prompts, the cloud server needs the file containing those prompts.

- open(..., "rb"): Opens the JSONL file containing your 1,000 prompts in "read-binary" mode. This is required for secure file transmission over the internet.

- groq.files.create(...): This pushes the physical file from your local hard drive into Groq's cloud storage.

- purpose="batch": A security and routing flag. It tells Groq's servers, "Do not use this file for fine-tuning or embeddings; put it in the specific storage bucket reserved for batch processing."

```python
file_id = response.id
file_id
```
Once a file is uploaded to AWS, GCP, or Groq, it no longer cares about your local filename (0_1000.jsonl). The cloud assigns it a unique, encrypted alphanumeric string (a UUID). You must capture this file_id because it is the only way to point the LLM to your specific data in the next step.

```python
response = groq.batches.create(completion_window="24h", endpoint="/v1/chat/completions", input_file_id=file_id)
response
```
This is the actual trigger that starts the AI processing. It creates a job ticket in Groq's system queue.

- endpoint="/v1/chat/completions": Tells the system what kind of AI model to use. You are telling it to run standard text-generation (chat), not audio transcription or image generation.

- input_file_id=file_id: Connects the job ticket to the file you uploaded in Step 1.

- completion_window="24h": The core feature of batching. You are telling Groq, "I am in no rush. Run these 1,000 prompts whenever your servers have downtime over the next 24 hours." This flexibility is why batch processing is significantly cheaper than real-time API calls.
```python

result = groq.batches.retrieve(response.id)
result
```
Because you gave the server 24 hours to finish, the code doesn't just freeze and wait (like a standard API call). The job runs asynchronously.

- batches.retrieve(response.id): This acts like checking a FedEx tracking number. You ping the server with the Job ID to see if the status is "queued", "in_progress", or "completed". If you were building a real app, you would put this in a loop that checks the status every 10 minutes.

```python
response = groq.files.content(result.output_file_id)
response.write_to_file("jsonl/batch_results.jsonl")
```
Once the job status is "completed", Groq generates a brand new JSONL file containing all the LLM's answers.

- result.output_file_id: The server gives you a new UUID for the answers file.

- groq.files.content(...): You request the contents of that specific file.

- .write_to_file(...): You pull the data stream from the cloud and write it back onto your local hard drive as a new file named batch_results.jsonl.

```python
with open("jsonl/batch_results.jsonl", "r") as f:
    for line in f:
        json_line = json.loads(line)
        id = int(json_line["custom_id"])
        summary = json_line["response"]["body"]["choices"][0]["message"]["content"]
        items[id].summary = summary
```
Now you have a local file filled with AI-generated summaries, but you need to attach those summaries to the specific Item objects currently sitting in your computer's RAM (the items list).

- The Async Problem: When the cloud processes a batch file, it does not guarantee that it will answer question #10 before question #11. The results come back totally out of order based on how the servers distributed the work.

- json_line["custom_id"]: When the original file was uploaded, Ed assigned a custom_id to every prompt that perfectly matched the item's index number in his local Python list.

- items[id].summary = summary: Because we have that custom_id, it doesn't matter what order the cloud sends the answers back. The code looks at the ID, jumps to that exact slot in the local items array, navigates deep into the JSON response to grab the actual text content, and permanently saves it into the item's .summary attribute.

### The Exact steps followed above are in batch.py file:

- Divides items into groups of 1,000
- Kicks off batches for each
- Allows us to monitor and collect the results when complete

## COSTS

Using Groq - this cost under 1-2 dollars for the Lite dataset and under 30 dollars for the big dataset.

Skipping doing the below code due to Cost Constraints

```python
Batch.create(items, LITE_MODE)
Batch.run()
Batch.fetch()

for index, item in enumerate(items):
    if not item.summary:
        print(index)

print(items[10234].summary)

# Remove the fields that we don't need in the hub

for item in items:
    item.full = None
    item.id = None
```